<div style="background-color: #F4F6F7; padding: 20px; border-radius: 8px; font-family: 'Arial', sans-serif; color: #2E4053;">
  <!-- Logo centrado -->
  <div style="text-align: center; margin-bottom: 15px;">
    <img src="https://drive.google.com/uc?export=view&id=1kjzXfjTiieYAd4azw5bh4UfEg91gUIdh" alt="Logo Institucional" width="250" />
  </div>

  <!-- Título principal -->
  <h1 style="font-size: 36px; font-weight: bold; text-align: right;">
    <strong>Conversión de las imágenes multiespectrales del LANDSAT 7 de formato HDF a PDF</strong>
  </h1>

  <!-- Datos del estudiante -->
  <p style="font-size: 22px; text-align: center; margin: 5px 0;">
    <strong>Pereira, 2025 </strong>  &nbsp;|&nbsp;
    <strong>Institución:</strong> Universidad Tecnológica de Pereira  &nbsp;|&nbsp;
    <strong>Grupo de investigación en Automática </strong>
  </p>

  <hr style="border: 1px solid #ABB2B9; margin: 10px 0;" />

  <!-- Descripción breve -->
  <p style="font-size: 18px; text-align: center; font-style: italic; margin: 5px 0;">
    Este código busca archivos HDF en una carpeta, toma la información de sus bandas y crea para cada archivo una imagen organizada en varios cuadros donde se ven hasta ocho bandas con buen contraste y una escala de colores. Luego guarda cada resultado como un PDF en otra carpeta, facilitando revisar visualmente los datos de forma rápida y ordenada.

  </p>

  <hr style="border: 1px solid #ABB2B9; margin: 10px 0;" />
</div>


# **En este cuaderno se plantea la extracción de las imágenes satelitales en formato HDF a PDF**

In [ ]:
import os
import glob
import numpy as np
import matplotlib.pyplot as plt
from pyhdf.SD import SD, SDC
from matplotlib.backends.backend_pdf import PdfPages

# --- Configuración ---
INPUT_DIR = "LANDSAT"                     # carpeta con .hdf
OUTPUT_DIR = "LANDSAT_BAND_PDFS"          # donde se guardan los PDFs
FIGSIZE = (16, 8)                         # tamaño figura en pulgadas
GRID_SHAPE = (2, 4)                       # filas, columnas (ajusta si quieres otra disposición)
CMAP = "inferno"                          # colormap
PERCENTILE_LOW, PERCENTILE_HIGH = 2, 98   # percentiles para estiramiento
DPI = 300                                 # resolución para raster contenidos en PDF

os.makedirs(OUTPUT_DIR, exist_ok=True)

# Encuentra todos los archivos HDF en la carpeta
hdf_files = sorted(glob.glob(os.path.join(INPUT_DIR, "*.hdf")) + glob.glob(os.path.join(INPUT_DIR, "*.HDF")))
if not hdf_files:
    raise FileNotFoundError(f"No se encontraron archivos .hdf en {INPUT_DIR}")

def read_datasets_from_hdf4(path):
    """Devuelve diccionario {nombre_dataset: numpy_array} usando pyhdf.SD"""
    hdf = SD(path, SDC.READ)
    datasets = hdf.datasets()  # dict
    bandas = {}
    # ordenaremos por la lista de keys para reproducibilidad
    keys = list(datasets.keys())
    for k in keys:
        try:
            s = hdf.select(k)
            arr = s[:].astype(np.float32)  # convierto a float para poder manejar NaN y escalas
            # reemplazar valores fill o extremos si vienen como -9999 u otro (no asumimos uno concreto)
            # No los tocamos por defecto; si conoces fill_value puedes aplicar mask aquí.
            bandas[k] = arr
        except Exception as e:
            print(f"  ⚠️ No se pudo leer dataset {k} en {os.path.basename(path)}: {e}")
    return bandas

def robust_vmin_vmax(arr, low_pct=PERCENTILE_LOW, high_pct=PERCENTILE_HIGH):
    """Calcula vmin/vmax ignorando NaNs e infinitos"""
    a = arr.copy().ravel()
    a = a[np.isfinite(a)]
    if a.size == 0:
        return 0, 1
    vmin = np.percentile(a, low_pct)
    vmax = np.percentile(a, high_pct)
    if vmin == vmax:
        # fallback
        vmin, vmax = a.min(), a.max()
        if vmin == vmax:
            vmax = vmin + 1.0
    return vmin, vmax

# --- Procesar cada archivo HDF ---
for hdf_path in hdf_files:
    base = os.path.splitext(os.path.basename(hdf_path))[0]
    print(f"\nProcesando: {base} ...")
    bandas_dict = read_datasets_from_hdf4(hdf_path)
    if not bandas_dict:
        print(f"  ⚠️ No hay datasets leíbles en {base}. Saltando.")
        continue

    # Queremos mostrar hasta 8 subplots (2x4). Tomamos los primeros 8 datasets encontrados.
    keys = list(bandas_dict.keys())
    n_to_plot = GRID_SHAPE[0] * GRID_SHAPE[1]
    keys_to_plot = keys[:n_to_plot]

    # Crear figura
    fig, axes = plt.subplots(GRID_SHAPE[0], GRID_SHAPE[1], figsize=FIGSIZE)
    axes = axes.ravel()

    for i, ax in enumerate(axes):
        ax.axis("off")  # por defecto oculto, luego si hay dato lo dibujamos
        if i < len(keys_to_plot):
            name = keys_to_plot[i]
            data = bandas_dict[name]
            # enmascarar nodata si aparecen valores extremos (opcional)
            # data = np.ma.masked_invalid(data)

            # calcular vmin/vmax robusto
            vmin, vmax = robust_vmin_vmax(data)

            im = ax.imshow(data, cmap=CMAP, vmin=vmin, vmax=vmax)
            ax.set_title(f"{name}", fontsize=9)
            ax.axis("off")
            # colorbar pequeña al lado (ajustada)
            cbar = plt.colorbar(im, ax=ax, fraction=0.045, pad=0.02)
            cbar.ax.tick_params(labelsize=6)
        else:
            ax.set_visible(False)

    plt.suptitle(f"Bandas - {base}", fontsize=14)
    plt.tight_layout(rect=[0, 0, 1, 0.96])

    # Guardar a PDF alto calidad (vectorial; dpi usado para imágenes raster incluidas)
    out_pdf = os.path.join(OUTPUT_DIR, f"{base}_bands.pdf")
    fig.savefig(out_pdf, format="pdf", dpi=DPI, bbox_inches="tight")
    plt.close(fig)  # libera memoria
    print(f"  ✅ Guardado: {out_pdf}")

print("\nProceso completado. Todos los PDFs están en:", OUTPUT_DIR)
